# 🧥 FashionVision v2 — Real Clothing Classifier (MobileNetV2)

Complete training pipeline optimized for **Google Colab (T4 GPU)**.

### Pipeline Stages:
1. **Environment Setup** — Verify T4 GPU acceleration
2. **Repository Sync** — Clone repo and pull latest fixes
3. **Kaggle Dataset Download** — Streamlined download of `agrigorev/clothing-dataset-full`
4. **Data Preparation** — Multi-threaded filtering, stratified/group splitting, resizing to 224×224
5. **Two-Stage Transfer Learning**:
   - *Stage 1*: Frozen MobileNetV2 base, train classification head (Adam, LR=1e-3)
   - *Stage 2*: Unfreeze top 30 layers, fine-tune with low learning rate (Adam, LR=1e-5)
6. **Evaluation on Test Split** — Accuracy, macro F1, confusion matrix, misclassified photos
7. **Download Artifacts** — Export `models/` and `results/` as a single `.zip`

> ⚡ **Check First:** Ensure runtime is set to GPU: `Runtime > Change runtime type > T4 GPU`

---
## 1. Verify GPU & Install Dependencies

In [1]:
import tensorflow as tf
import sys

print(f"TensorFlow version : {tf.__version__}")
print(f"Python version     : {sys.version}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU detected: {[g.name for g in gpus]}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    print("   Memory growth configured.")
else:
    print("\n⚠️ No GPU detected! Please enable GPU via:\n   Runtime > Change runtime type > T4 GPU")

TensorFlow version : 2.21.0
Python version     : 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

✅ GPU detected: ['/physical_device:GPU:0']
   Memory growth configured.


In [2]:
# Install lightweight packages required for the workflow
# Only kagglehub needs installation; scikit-learn is pre-installed
!pip install -q kagglehub

---
## 2. Clone Repository & Pull Latest Code

In [3]:
import os
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AdityaB197/FashionVision.git'
if Path('/kaggle/working').exists():
    REPO_DIR = Path('/kaggle/working/FashionVision')
else:
    REPO_DIR = Path('/content/FashionVision')

if REPO_DIR.exists():
    print(f'Repo already cloned at {REPO_DIR}. Pulling latest changes...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print(f'Cloning repo from {REPO_URL}...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Current working directory: {os.getcwd()}')

# Verify local modules import properly
from src.data import load_datasets, compute_class_weights
from src.model import build_real_model, enable_fine_tuning
from src.inference import load_real_artifacts, prepare_real_image, predict_top_k_real
from src.reporting import compute_metrics, plot_confusion_matrix, save_json
print('✅ All FashionVision v2 source modules loaded successfully.')


Repo already cloned at /content/FashionVision. Pulling latest changes...
Current working directory: /content/FashionVision
✅ All FashionVision v2 source modules loaded successfully.


---
## 3. Kaggle Authentication & Dataset Download

Provide credentials using **either**:
- **Option A (Recommended)**: Colab Secrets (Keys: `KAGGLE_USERNAME`, `KAGGLE_KEY` in the left sidebar 🔑)
- **Option B**: Upload your `kaggle.json` API token file

In [4]:
import os

# Attempt loading from Colab Secrets
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials loaded from Colab Secrets.")
except Exception:
    print("ℹ️ Colab Secrets not found. Checking ~/.kaggle/kaggle.json...")
    kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kaggle_json_path):
        print("Please upload your kaggle.json file below:")
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
            with open(kaggle_json_path, 'wb') as f:
                f.write(uploaded['kaggle.json'])
            os.chmod(kaggle_json_path, 0o600)
            print("✅ kaggle.json installed successfully.")
        else:
            print("⚠️ No kaggle.json uploaded. Download will proceed; login prompt may appear.")
    else:
        print("✅ Existing kaggle.json detected.")

ℹ️ Colab Secrets not found. Checking ~/.kaggle/kaggle.json...
✅ Existing kaggle.json detected.


In [5]:
import kagglehub

print("Downloading agrigorev/clothing-dataset-full (~3-4 GB)... This takes ~2-4 minutes on Colab.")
raw_dataset_path = kagglehub.dataset_download("agrigorev/clothing-dataset-full")
print(f"\n✅ Raw dataset downloaded to: {raw_dataset_path}")

Using Colab cache for faster access to the 'clothing-dataset-full' dataset.

✅ Raw dataset downloaded to: /kaggle/input/clothing-dataset-full


---
## 4. Prepare Data (Filter, Split, Resize)

This filters uninformative categories, partitions into 70% train / 15% val / 15% test,
and resizes photos to 224×224 using multi-threaded LANCZOS interpolation.

In [6]:
import subprocess
import sys
import os
from pathlib import Path
import kagglehub

# Auto-locate repository root
possible_roots = [
    Path.cwd(),
    Path('/content/FashionVision'),
    Path('/kaggle/working/FashionVision'),
    Path.cwd() / 'FashionVision'
]
repo_root = None
for r in possible_roots:
    if (r / 'scripts' / 'prepare_data.py').is_file():
        repo_root = r
        break

if repo_root:
    os.chdir(str(repo_root))
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    script_path = repo_root / 'scripts' / 'prepare_data.py'
else:
    script_path = Path('scripts/prepare_data.py')

# Ensure dataset path is resolved (Colab or Kaggle)
if 'raw_dataset_path' not in globals() or not raw_dataset_path:
    if Path('/kaggle/input/clothing-dataset-full').exists():
        raw_dataset_path = '/kaggle/input/clothing-dataset-full'
    else:
        print('Resolving Kaggle dataset path with kagglehub...')
        raw_dataset_path = kagglehub.dataset_download('agrigorev/clothing-dataset-full')

PREPARED_DIR = 'data/clothing_prepared'

cmd = [
    sys.executable, str(script_path),
    '--source-dir', str(raw_dataset_path),
    '--output-dir', PREPARED_DIR,
    '--min-images-per-class', '100',
    '--img-size', '224',
    '--seed', '42',
    '--skip-if-exists'
]

print('Executing:', ' '.join(cmd))
result = subprocess.run(cmd, check=True)
print('\n✅ Data preparation finished.')


Executing: /usr/bin/python3 /content/FashionVision/scripts/prepare_data.py --source-dir /kaggle/input/clothing-dataset-full --output-dir data/clothing_prepared --min-images-per-class 100 --img-size 224 --seed 42 --skip-if-exists

✅ Data preparation finished.


In [7]:
# Verify split summary
import json
from pathlib import Path

summary_file = Path(PREPARED_DIR) / "split_summary.json"
with open(summary_file, "r", encoding="utf-8") as f:
    summary = json.load(f)

print(f"Classes ({len(summary['class_names'])}): {summary['class_names']}")
print(f"Total images processed: {summary['total_images_processed']}")
print(f"Split method: {summary['split_method']}")
for split, counts in summary['counts_per_split'].items():
    print(f"  {split:5s}: {sum(counts.values()):4d} images")

Classes (14): ['Blazer', 'Dress', 'Hat', 'Hoodie', 'Longsleeve', 'Outwear', 'Pants', 'Polo', 'Shirt', 'Shoes', 'Shorts', 'Skirt', 'T-Shirt', 'Undershirt']
Total images processed: 4961
Split method: group_aware_by_sender_id
  train: 3879 images
  val  :  679 images
  test :  403 images


---
## 5. Train MobileNetV2 (Two-Stage Transfer Learning)

Optimizations applied:
- Mixed precision (`mixed_float16`) for ~2× GPU speedup on T4
- Efficient `tf.data.prefetch(AUTOTUNE)` streaming (avoids RAM caching crashes)
- Balanced class weights to handle natural class distribution skew

In [8]:
import subprocess
import sys

train_cmd = [
    sys.executable, "scripts/train_model.py",
    "--data-dir", PREPARED_DIR,
    "--output-dir", ".",
    "--epochs-head", "10",
    "--epochs-finetune", "10",
    "--batch-size", "32",
    "--patience", "4",
    "--seed", "42"
]

print("Starting training pipeline:", " ".join(train_cmd))
result = subprocess.run(train_cmd, check=True)
print("\n✅ Training completed successfully!")

Starting training pipeline: /usr/bin/python3 scripts/train_model.py --data-dir data/clothing_prepared --output-dir . --epochs-head 10 --epochs-finetune 10 --batch-size 32 --patience 4 --seed 42

✅ Training completed successfully!


---
## 6. Evaluate Model on Test Split

In [9]:
eval_cmd = [
    sys.executable, "scripts/evaluate_model.py",
    "--data-dir", PREPARED_DIR,
    "--output-dir", "."
]

print("Running evaluation on held-out test split:", " ".join(eval_cmd))
result = subprocess.run(eval_cmd, check=True)
print("\n✅ Evaluation completed.")

Running evaluation on held-out test split: /usr/bin/python3 scripts/evaluate_model.py --data-dir data/clothing_prepared --output-dir .

✅ Evaluation completed.


In [10]:
# Display test metrics
import json
from pathlib import Path

with open("results/metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("=" * 50)
print(f"Test Accuracy : {metrics['accuracy'] * 100:.2f}%")
print(f"Macro F1      : {metrics['macro_f1']:.4f}")
print(f"Weighted F1   : {metrics['weighted_f1']:.4f}")
print("=" * 50)

Test Accuracy : 83.37%
Macro F1      : 0.6873
Weighted F1   : 0.8402


In [11]:
# Display evaluation charts
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for title, path in [
    ("Training Trajectory (Head & Fine-Tuning)", "results/training_curves.png"),
    ("Confusion Matrix (Real Clothing Test Split)", "results/confusion_matrix.png"),
    ("Misclassified Test Samples", "results/misclassified_examples.png"),
]:
    if Path(path).is_file():
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

---
## 7. Package & Download Artifacts

Packages `models/` and `results/` into a single zip file for direct browser download.
After downloading, extract the zip into your local `ML mini Project` folder.

In [12]:
import shutil
import os
from pathlib import Path
from google.colab import files

ZIP_NAME = "fashionvision_v2_artifacts"
STAGING = Path("/content/staging_v2")

if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir()

shutil.copytree("models", STAGING / "models")
shutil.copytree("results", STAGING / "results")

zip_file = shutil.make_archive(f"/content/{ZIP_NAME}", 'zip', STAGING)
shutil.rmtree(STAGING)

size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f"✅ Artifact archive created: {zip_file} ({size_mb:.1f} MB)")

print("\nStarting browser download...")
files.download(zip_file)

✅ Artifact archive created: /content/fashionvision_v2_artifacts.zip (31.2 MB)

Starting browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>